In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from sklearn.metrics import r2_score

import Functions

In [85]:
df_stats = pd.read_csv('/home/frank/Desktop/f.chiapperino_local/VALENCIA/SCUOLA/App/Documents/csv/TimeSeries_Coherencia.csv', index_col='Code')
df_stats['Date'] = pd.to_datetime(df_stats['Date'])
display(df_stats)

,Date_Band,mean,std,Date,Band,Name,Type
Code,,,,,,,
1841224,20170104_VH,0.846021,0.060364,2017-01-04,VH,DB_A_C,Potatoes
1841224,20170104_VV,0.912519,0.032522,2017-01-04,VV,DB_A_C,Potatoes
1841224,20170110_VH,0.470780,0.095106,2017-01-10,VH,DB_A_C,Potatoes
1841224,20170110_VV,0.660484,0.081345,2017-01-10,VV,DB_A_C,Potatoes
1841224,20170116_VH,0.311812,0.113724,2017-01-16,VH,DB_A_C,Potatoes
...,...,...,...,...,...,...,...
1841225,20171212_VV,0.191203,0.106592,2017-12-12,VV,DB_SB1_C,Beets
1841225,20171218_VH,0.916552,0.026221,2017-12-18,VH,DB_SB1_C,Beets
1841225,20171218_VV,0.957047,0.016386,2017-12-18,VV,DB_SB1_C,Beets


In [86]:
df_stats.drop(df_stats[df_stats['mean'] == 0].index,inplace=True)
display(pd.unique(df_stats.index))

Index([1841224, 1936134, 2088262, 1576641, 1697689, 1896343, 2278835, 1601502,
       1936133, 1824038, 1545384, 1553694, 2081268, 2081267, 2081887, 1767881,
       1698168, 1841223, 1936135, 2041694, 1697690, 1631664, 1841225],
      dtype='int64', name='Code')

In [92]:
df_height = pd.read_csv(r'/home/frank/Desktop/f.chiapperino_local/VALENCIA/SCUOLA/App/Documents/csv/Averaged_crop_height.csv', header=0)
df_height=df_height.set_index(df_height['Code'])
df_height = df_height.drop(columns=['Code'])
display(df_height)

,2017-05-18 00:00:00,2017-05-23 00:00:00,2017-05-30 00:00:00,2017-06-08 00:00:00,2017-06-15 00:00:00,2017-06-21 00:00:00,2017-06-27 00:00:00,2017-07-03 00:00:00,2017-07-06 00:00:00,2017-07-13 00:00:00,2017-07-17 00:00:00,2017-07-27 00:00:00,2017-08-02 00:00:00,2017-08-11 00:00:00,2017-08-17 00:00:00,2017-08-23 00:00:00,2017-09-01 00:00:00,2017-09-07 00:00:00,2017-09-15 00:00:00
Code,,,,,,,,,,,,,,,,,,,
1545384.0,NaN,NaN,NaN,NaN,NaN,59.5,67.5,84.0,NaN,83.50,85.00,87.5,71.0,75.00,60.00,52.50,NaN,53.50,25.0
1553694.0,NaN,NaN,NaN,NaN,NaN,53.0,56.5,57.5,NaN,75.50,75.00,71.5,73.5,66.00,82.50,77.50,NaN,79.00,NaN
1576641.0,NaN,NaN,NaN,NaN,90.0,NaN,NaN,NaN,77.0,NaN,90.00,NaN,10.0,NaN,NaN,NaN,15.0,NaN,NaN
1601502.0,NaN,NaN,NaN,NaN,NaN,42.5,51.5,58.0,NaN,NaN,59.00,62.0,65.0,64.00,70.00,78.50,NaN,77.50,NaN
1631664.0,21.0,23.0,38.0,NaN,9.0,14.0,20.0,20.5,NaN,31.00,37.75,NaN,24.0,28.25,17.00,23.00,NaN,23.75,20.0
1697689.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.00,15.00,NaN,22.0,NaN,20.00,NaN,32.5,NaN,NaN
1697690.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.00,18.00,NaN,30.0,NaN,25.00,NaN,35.0,NaN,NaN
1697691.0,NaN,NaN,NaN,NaN,NaN,NaN,58.5,NaN,NaN,40.00,45.00,NaN,45.0,NaN,55.00,NaN,30.0,NaN,NaN
1698168.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,240.00,250.00,NaN,270.0,NaN,NaN,NaN,NaN,NaN,NaN


In [93]:
df_rain = pd.read_csv('/home/frank/Desktop/f.chiapperino_local/VALENCIA/SCUOLA/App/Documents/csv/rainfall_week.csv', header=0)
display(df_rain)

,date,prec_mm
0,2017-01-04,23.0
1,2017-01-10,58.8
2,2017-01-16,0.0
3,2017-01-22,7.8
4,2017-01-28,20.1
...,...,...
58,2017-12-18,5.7
59,2017-12-24,37.8
60,2017-12-30,112.0
61,2018-01-05,6.3


In [108]:
# Filtra i dati per la parcella 1553694
df_growth_beets = df_height.loc[1553694].dropna()

# 1. Creazione di un indice giornaliero per non perdere i "denti"
daily_index = pd.date_range(start=df_growth_beets.index.min(), 
                             end=df_growth_beets.index.max(), freq='D')
display(type(daily_index))


pandas.DatetimeIndex

In [107]:
# 2. Interpolazione PCHIP (preserva meglio la forma a dente di sega rispetto alla lineare)
from scipy.interpolate import pchip_interpolate
growth_interp = df_growth_beets.reindex(daily_index).interpolate(method='pchip')
print(growth_interp.iloc[5:10])

2017-06-26   NaN
2017-06-27   NaN
2017-06-28   NaN
2017-06-29   NaN
2017-06-30   NaN
Freq: D, Name: 1553694.0, dtype: float64


regressione

In [ ]:
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

X = 